In [2]:
import numpy as np


def _extract_extrema(sigma):
    # Reduce a stress history to its sequence of local peaks and valleys
    sigma = np.asarray(sigma, dtype=float)
    if len(sigma) < 2:
        return sigma.copy()
    keep = np.concatenate(([True], np.diff(sigma) != 0)) # this marks where values actually change
    s = sigma[keep]
    if len(s) < 3:
        return s
    diffs = np.diff(s)
    sign_changes = np.diff(np.sign(diffs))
    extrema_idx = np.where(sign_changes != 0)[0] + 1
    return np.concatenate(([s[0]], s[extrema_idx], [s[-1]]))


def rainflow_cycles(sigma):
    """
    based on ASTM E1049-85 aka four point method
    
    returns cycles as tuples
    """
    extrema = _extract_extrema(sigma)
    if len(extrema) < 2:
        return []

    cycles = []
    stack = []

    for x in extrema:
        stack.append(x)
        while len(stack) >= 4:
            S1, S2, S3, S4 = stack[-4], stack[-3], stack[-2], stack[-1]
            r12 = abs(S1 - S2)
            r23 = abs(S2 - S3)
            r34 = abs(S3 - S4)
            if r23 <= r12 and r23 <= r34:
                cycles.append((r23, 0.5 * (S2 + S3), 1.0)) ## saves the range (abs r23), mean, and counts it as full count
                del stack[-3:-1]
            else:
                break

    for i in range(len(stack) - 1):
        rng = abs(stack[i] - stack[i + 1])
        mean = 0.5 * (stack[i] + stack[i + 1])
        cycles.append((rng, mean, 0.5)) # residual half cycles

    return cycles


def rainflow_to_arrays(cycles):
    if not cycles:
        return (np.array([]), np.array([]), np.array([]))
    arr = np.array(cycles, dtype=float)
    return arr[:, 0], arr[:, 1], arr[:, 2]
